import yfinance as yf
print(yf.__version__)

In [ ]:
# OHLC + Volume Feature Engineering

In [ ]:
# 1. Data Integrity

In [13]:
## 1.1 Pobierz dane

In [14]:
import yfinance as yf
import pandas as pd

# Parametry
TICKER = "^GSPC"
START_DATE = "1900-01-01"
END_DATE = None  # None = do dziś
INTERVAL = "1d"  # daily candles

# Pobranie danych
df = yf.download(
    tickers=TICKER,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    auto_adjust=False,
    progress=False
)

# Jeśli kolumny są MultiIndex (np. ('Open','^GSPC')), spłaszczamy
if isinstance(df.columns, pd.MultiIndex):
    # najczęściej: pierwszy poziom to nazwa pola (Open/High/...)
    # a drugi to ticker. Bierzemy pierwszy.
    df.columns = [c[0] for c in df.columns]

# Standaryzacja nazw kolumn
df = df.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Adj Close": "adj_close",
    "Volume": "volume"
})

# Usunięcie wierszy z brakami
df = df.dropna()

print(df.head())
print(df.tail())
print(df.info())


            adj_close      close       high        low       open  volume
Date                                                                     
1927-12-30  17.660000  17.660000  17.660000  17.660000  17.660000       0
1928-01-03  17.760000  17.760000  17.760000  17.760000  17.760000       0
1928-01-04  17.719999  17.719999  17.719999  17.719999  17.719999       0
1928-01-05  17.549999  17.549999  17.549999  17.549999  17.549999       0
1928-01-06  17.660000  17.660000  17.660000  17.660000  17.660000       0
              adj_close        close         high          low         open  \
Date                                                                          
2026-01-26  6950.229980  6950.229980  6964.660156  6921.600098  6923.229980   
2026-01-27  6978.600098  6978.600098  6988.819824  6958.830078  6965.959961   
2026-01-28  6978.029785  6978.029785  7002.279785  6963.459961  7002.000000   
2026-01-29  6969.009766  6969.009766  6992.839844  6870.799805  6977.740234   
2026-01-

In [15]:
### 1.1.1 Utnij dane które są nieistotne - wykazano po sprawdzeniu danych

In [16]:
df = df[df.index >= "1952-01-01"].copy()

In [17]:
### 1.1.2 Wymuszenie polityki timestampów

In [30]:
import pandas as pd

def enforce_daily_session_timestamps(df: pd.DataFrame, *, verbose: bool = True) -> pd.DataFrame:
    """
    Wymusza politykę timestampów:
    - DatetimeIndex
    - timezone-naive
    - normalized to session date (00:00)
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("Index must be pandas.DatetimeIndex")

    df = df.copy()
    idx_before = df.index

    had_tz = idx_before.tz is not None
    had_time_component = not (idx_before == idx_before.normalize()).all()

    if had_tz:
        df.index = df.index.tz_convert(None)

    df.index = df.index.normalize()

    if verbose:
        print("🕒 enforce_daily_session_timestamps:")
        print(f"   - timezone removed: {had_tz}")
        print(f"   - time component removed: {had_time_component}")
        print("   - policy: daily session dates (naive)")

    return df

df = enforce_daily_session_timestamps(df)


🕒 enforce_daily_session_timestamps:
   - timezone removed: False
   - time component removed: False
   - policy: daily session dates (naive)


In [19]:
### 1.1.3 Correct timestamps – twarda walidacja

In [31]:
import pandas as pd

def check_correct_timestamps(df: pd.DataFrame) -> None:
    """
    Waliduje poprawność timestampów dla daily OHLC.
    Rzuca ValueError z czytelnym komunikatem przy pierwszym błędzie.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("Index is not a pandas.DatetimeIndex")

    idx = df.index

    if idx.tz is not None:
        raise ValueError("Index has timezone info; expected timezone-naive daily timestamps")

    if not idx.is_monotonic_increasing:
        raise ValueError("Timestamps are not sorted (not monotonic increasing)")

    if not idx.is_unique:
        raise ValueError("Duplicate timestamps detected")

    if not (idx == idx.normalize()).all():
        bad = idx[idx != idx.normalize()]
        raise ValueError(
            f"Timestamps contain time component; examples: {bad[:5].tolist()}"
        )

    print("✅ Correct timestamps: PASS")

check_correct_timestamps(df)

✅ Correct timestamps: PASS


In [32]:
## 1.2. Sprawdź dane

In [33]:
### 1.2.1 Indeks czasowy
assert isinstance(df.index, pd.DatetimeIndex)
assert df.index.is_monotonic_increasing
assert df.index.is_unique

### 1.2.2 Logika OHLC
assert (df["high"] >= df[["open", "close"]].max(axis=1)).all()
assert (df["low"]  <= df[["open", "close"]].min(axis=1)).all()
assert (df["high"] >= df["low"]).all()

### 1.2.3 Wolumen
assert (df["volume"] >= 0).all()

print("✅ Data Integrity: podstawowe testy zaliczone")


✅ Data Integrity: podstawowe testy zaliczone


In [34]:
import pandas as pd

def check_session_gaps_daily(df: pd.DataFrame, max_gap_days: int = 4):
    """
    Dla danych dziennych wykrywa podejrzane luki między kolejnymi świecami.
    Domyślnie >4 dni jest podejrzane (bo weekend to 2 dni, długi weekend ~3-4).
    """
    assert isinstance(df.index, pd.DatetimeIndex)

    idx = df.index.sort_values().normalize()
    deltas = idx.to_series().diff().dropna()

    suspicious = deltas[deltas > pd.Timedelta(days=max_gap_days)]
    return {
        "n_rows": len(df),
        "max_gap": deltas.max(),
        "n_suspicious_gaps": len(suspicious),
        "suspicious_gaps": suspicious.head(20),  # pokaże daty i wielkość przerwy
    }

# użycie:
report = check_session_gaps_daily(df, max_gap_days=4)
print(report)


{'n_rows': 18643, 'max_gap': Timedelta('7 days 00:00:00'), 'n_suspicious_gaps': 7, 'suspicious_gaps': Date
1956-12-26   5 days
1958-12-29   5 days
1961-05-31   5 days
1968-07-08   5 days
2001-09-17   7 days
2007-01-03   5 days
2012-10-31   5 days
Name: Date, dtype: timedelta64[ns]}


In [35]:
import pandas as pd
import pandas_market_calendars as mcal

def check_continuous_sessions_with_calendar(
    df: pd.DataFrame,
    calendar_name: str = "NYSE",
    *,
    weekdays: tuple[int, ...] = (0, 1, 2, 3, 4),  # 0=Mon ... 6=Sun
    normalize_index: bool = True,
):
    """
    Sprawdza, czy df ma świecę dla każdej oczekiwanej sesji z kalendarza giełdowego.
    
    Parametry:
    - weekdays: które dni tygodnia uznajemy za "potencjalne sesje".
      Domyślnie (0..4) => poniedziałek–piątek.
      Jeśli chcesz dopuścić soboty: (0,1,2,3,4,5)
    - normalize_index: jeśli True, porównuje po datach (bez godzin).
    
    Zwraca słownik z liczbą braków i podglądem brakujących dat.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("df.index musi być pandas.DatetimeIndex")

    idx = df.index
    if normalize_index:
        # dzienne dane z yfinance zwykle są bez TZ; normalizacja to bezpieczny standard
        idx = pd.DatetimeIndex(idx).normalize()

    idx = idx.sort_values()
    idx = idx[~idx.duplicated(keep="first")]

    cal = mcal.get_calendar(calendar_name)
    start = idx.min().date()
    end = idx.max().date()

    schedule = cal.schedule(start_date=start, end_date=end)

    expected_sessions = pd.DatetimeIndex(schedule.index)
    if normalize_index:
        expected_sessions = expected_sessions.normalize()

    # 🔑 Filtrujemy kalendarz do wybranych dni tygodnia (np. Mon–Fri)
    expected_sessions = expected_sessions[expected_sessions.weekday.isin(weekdays)]

    missing_sessions = expected_sessions.difference(idx)
    extra_sessions = idx.difference(expected_sessions)

    # Dodatkowe info diagnostyczne
    return {
        "calendar": calendar_name,
        "weekdays": weekdays,
        "start": str(start),
        "end": str(end),
        "n_expected_sessions": len(expected_sessions),
        "n_actual_sessions": len(idx),
        "n_missing_sessions": len(missing_sessions),
        "missing_sessions_head": missing_sessions[:20],
        "n_extra_sessions": len(extra_sessions),
        "extra_sessions_head": extra_sessions[:20],
    }

report = check_continuous_sessions_with_calendar(df, "NYSE", weekdays=(0,1,2,3,4))
print(report)


{'calendar': 'NYSE', 'weekdays': (0, 1, 2, 3, 4), 'start': '1952-01-02', 'end': '2026-01-30', 'n_expected_sessions': 18643, 'n_actual_sessions': 18643, 'n_missing_sessions': 0, 'missing_sessions_head': DatetimeIndex([], dtype='datetime64[ns]', freq=None), 'n_extra_sessions': 0, 'extra_sessions_head': DatetimeIndex([], dtype='datetime64[ns]', freq=None)}


In [40]:
assert report["n_missing_sessions"] == 0

In [ ]:
# 2. Price Transforms

In [41]:
import numpy as np
import pandas as pd

def compute_log_return(close: pd.Series) -> pd.Series:
    close = close.astype(float)
    # log(close_t) - log(close_{t-1})
    return np.log(close).diff()


In [42]:
from sklearn.base import BaseEstimator, TransformerMixin

class LogReturnTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, close_col: str = "close", output_col: str = "log_return", drop_na: bool = True):
        self.close_col = close_col
        self.output_col = output_col
        self.drop_na = drop_na

    def fit(self, X, y=None):
        # Stateless transformer: niczego nie uczymy
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")

        if self.close_col not in X.columns:
            raise ValueError(f"Missing required column: '{self.close_col}'")

        out = pd.DataFrame(index=X.index)
        out[self.output_col] = compute_log_return(X[self.close_col])

        # Pierwszy wiersz będzie NaN (bo diff)
        if self.drop_na:
            out = out.dropna()

        return out


In [43]:
# zakładam, że masz df z kolumnami: open/high/low/close/volume
lrt = LogReturnTransformer(close_col="close", output_col="log_return", drop_na=False)

logret = lrt.fit_transform(df)

print(logret.head(5))
print(logret.describe())

# Sanity checks
assert "log_return" in logret.columns
# 1) Pierwsza świeca ma NaN (brak t-1)
assert pd.isna(logret["log_return"].iloc[0])
# 2) Reszta powinna być skończona (o ile nie masz close <= 0)
assert np.isfinite(logret["log_return"].iloc[1:]).all()

print("✅ LogReturnTransformer: PASS")


            log_return
Date                  
1952-01-02         NaN
1952-01-03    0.003356
1952-01-04    0.001674
1952-01-07   -0.000418
1952-01-08   -0.003771
         log_return
count  18642.000000
mean       0.000304
std        0.010007
min       -0.228997
25%       -0.004079
50%        0.000490
75%        0.005064
max        0.109572
✅ LogReturnTransformer: PASS
